<a href="https://colab.research.google.com/github/19mddill/Deep_Learning_Notebooks/blob/main/training_DNN_Keras.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [172]:
import tensorflow as tf
import numpy as np

In [173]:
dense = tf.keras.layers.Dense(50,activation="relu",kernel_initializer = "he_normal")

In [174]:
he_ag_init = tf.keras.initializers.VarianceScaling(scale=2, mode="fan_avg", distribution="uniform")

In [175]:
dense = tf.keras.layers.Dense(50,activation="sigmoid",kernel_initializer=he_ag_init)

In [176]:
leaky_relu = tf.keras.layers.LeakyReLU(alpha=0.2)
dense = tf.keras.layers.Dense(50,activation=leaky_relu,kernel_initializer="he_normal")

In [177]:
model = tf.keras.Sequential([
    tf.keras.layers.Flatten(input_shape=[28,28]),
    tf.keras.layers.BatchNormalization(),
    tf.keras.layers.Dense(300,activation="relu",kernel_initializer="he_normal"),
    tf.keras.layers.BatchNormalization(),
    tf.keras.layers.Dense(100,activation="relu",kernel_initializer="he_normal"),
    tf.keras.layers.BatchNormalization(),
    tf.keras.layers.Dense(10,activation="softmax")
])

In [178]:
model.summary()

Model: "sequential_16"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ flatten_14 (Flatten)            │ (None, 784)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_20          │ (None, 784)            │         3,136 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_65 (Dense)                │ (None, 300)            │       235,500 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_21          │ (None, 300)            │         1,200 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_66 (Dense)                │ (None, 100)            │        30,100 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_22          │ (None, 100)            │           400 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_67 (Dense)                │ (None, 10)             │         1,010 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 271,346 (1.04 MB)

 Trainable params: 268,978 (1.03 MB)

 Non-trainable params: 2,368 (9.25 KB)

In [179]:
[(var.name,var.trainable) for var in model.layers[1].variables]

[('gamma', True),
 ('beta', True),
 ('moving_mean', False),
 ('moving_variance', False)]

In [180]:
model = tf.keras.Sequential([
    tf.keras.layers.Flatten(input_shape=[28,28]),
    tf.keras.layers.Dense(300,kernel_initializer="he_normal",use_bias=False),
    tf.keras.layers.BatchNormalization(),
    tf.keras.layers.Activation("relu"),
    tf.keras.layers.Dense(100,kernel_initializer="he_normal",use_bias=False),
    tf.keras.layers.BatchNormalization(),
    tf.keras.layers.Activation("relu"),
    tf.keras.layers.Dense(10,activation="softmax")
])

# **Resuing Pretrained Layer**


## **Fetching the MNIST**

In [181]:
fashion_mnnist = tf.keras.datasets.fashion_mnist.load_data()
class_names = ["T-shirt/top", "Trouser", "Pullover", "Dress", "Coat",
               "Sandal", "Shirt", "Sneaker", "Bag", "Ankle boot"]

In [182]:
(X_train_full,y_train_full),(X_test,y_test) = fashion_mnnist

In [183]:
X_train_full.shape,X_test.shape

((60000, 28, 28), (10000, 28, 28))

In [184]:
X_train, y_train = X_train_full[:-5000], y_train_full[:-5000]
X_valid, y_valid = X_train_full[-5000:], y_train_full[-5000:]

In [185]:
X_valid.shape

(5000, 28, 28)

In [186]:
X_train, X_valid, X_test = X_train / 255., X_valid / 255., X_test / 255.

In [187]:
pos_class_id = class_names.index("Pullover")
neg_class_id = class_names.index("T-shirt/top")
pos_class_id,neg_class_id


(2, 0)

In [188]:
y_train[455]

np.uint8(3)

# **test**

In [189]:
y = np.array([1,2,1,4,1,6,1,2]) #Blessed

mask_for_two = (y % 2) == 0

(y[mask_for_two] == 2).astype(np.float32)


array([1., 0., 0., 1.], dtype=float32)

## **Defining a new data splitter**

In [190]:
from types import new_class
def split_dataset(X,y):
  y_for_B = (y == pos_class_id) | (y == neg_class_id)
  y_A = y[~y_for_B] # (y != pos_class_id) & (y != neg_class_id)
  y_B = (y[y_for_B] == pos_class_id).astype(np.float32)
  old_class_ids = list(set(range(10))-set([neg_class_id,pos_class_id]))
  for old_class_id,new_class_id in zip(old_class_ids,range(8)):
    y_A[y_A == old_class_id] = new_class_id
  return ((X[~y_for_B],y_A),(X[y_for_B],y_B))

## **Splitting the data**

In [191]:
(X_train_A, y_train_A), (X_train_B, y_train_B) = split_dataset(X_train, y_train)
(X_valid_A, y_valid_A), (X_valid_B, y_valid_B) = split_dataset(X_valid, y_valid)
(X_test_A, y_test_A), (X_test_B, y_test_B) = split_dataset(X_test, y_test)

In [192]:
X_train_B = X_train_B[:200]
y_train_B = y_train_B[:200]

## **Execution**

In [193]:
tf.random.set_seed(42)

In [194]:
model_A = tf.keras.Sequential([
    tf.keras.layers.Flatten(input_shape=[28, 28]),
    tf.keras.layers.Dense(100, activation="relu",
                          kernel_initializer="he_normal"),
    tf.keras.layers.Dense(100, activation="relu",
                          kernel_initializer="he_normal"),
    tf.keras.layers.Dense(100, activation="relu",
                          kernel_initializer="he_normal"),
    tf.keras.layers.Dense(8, activation="softmax")
])

model_A.compile(loss="sparse_categorical_crossentropy",
                optimizer=tf.keras.optimizers.SGD(learning_rate=0.001,momentum=0.9),
                metrics=["accuracy"])


In [195]:
history = model_A.fit(X_train_A, y_train_A, epochs=20,
                      validation_data=(X_valid_A, y_valid_A))

Epoch 1/20
1376/1376 ━━━━━━━━━━━━━━━━━━━━ 5s 3ms/step - accuracy: 0.8228 - loss: 0.5284 - val_accuracy: 0.8789 - val_loss: 0.3308
Epoch 2/20
1376/1376 ━━━━━━━━━━━━━━━━━━━━ 6s 4ms/step - accuracy: 0.8933 - loss: 0.3053 - val_accuracy: 0.8935 - val_loss: 0.2848
Epoch 3/20
1376/1376 ━━━━━━━━━━━━━━━━━━━━ 4s 3ms/step - accuracy: 0.9064 - loss: 0.2678 - val_accuracy: 0.9067 - val_loss: 0.2632
Epoch 4/20
1376/1376 ━━━━━━━━━━━━━━━━━━━━ 5s 3ms/step - accuracy: 0.9138 - loss: 0.2451 - val_accuracy: 0.9110 - val_loss: 0.2496
Epoch 5/20
1376/1376 ━━━━━━━━━━━━━━━━━━━━ 5s 4ms/step - accuracy: 0.9195 - loss: 0.2289 - val_accuracy: 0.9150 - val_loss: 0.2401
Epoch 6/20
1376/1376 ━━━━━━━━━━━━━━━━━━━━ 4s 3ms/step - accuracy: 0.9238 - loss: 0.2158 - val_accuracy: 0.9180 - val_loss: 0.2325
Epoch 7/20
1376/1376 ━━━━━━━━━━━━━━━━━━━━ 5s 4ms/step - accuracy: 0.9270 - loss: 0.2048 - val_accuracy: 0.9200 - val_loss: 0.2270
Epoch 8/20
1376/1376 ━━━━━━━━━━━━━━━━━━━━ 9s 3ms/step - accuracy: 0.9303 - loss: 0.1955 - 

In [196]:
model_A.evaluate(X_test_A,y_test_A)

250/250 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - accuracy: 0.9190 - loss: 0.2379


[0.23790374398231506, 0.9190000295639038]

In [197]:
model_A.save("my_model_A.keras") #already done model by an expert.cant take all the layers of model_A.

In [198]:
tf.random.set_seed(42)

In [199]:
tf.random.set_seed(42)
model_B = tf.keras.Sequential([
    tf.keras.layers.Flatten(input_shape=[28, 28]),
    tf.keras.layers.Dense(100, activation="relu",
                          kernel_initializer="he_normal"),
    tf.keras.layers.Dense(100, activation="relu",
                          kernel_initializer="he_normal"),
    tf.keras.layers.Dense(100, activation="relu",
                          kernel_initializer="he_normal"),
    tf.keras.layers.Dense(1, activation="sigmoid")
])

model_B.compile(loss="binary_crossentropy",
                optimizer=tf.keras.optimizers.SGD(learning_rate=0.001),
                metrics=["accuracy"])

In [200]:
history = model_B.fit(X_train_B, y_train_B, epochs=20,
                      validation_data=(X_valid_B, y_valid_B))

Epoch 1/20
7/7 ━━━━━━━━━━━━━━━━━━━━ 1s 54ms/step - accuracy: 0.5550 - loss: 0.7469 - val_accuracy: 0.5153 - val_loss: 0.7463
Epoch 2/20
7/7 ━━━━━━━━━━━━━━━━━━━━ 0s 22ms/step - accuracy: 0.5550 - loss: 0.6953 - val_accuracy: 0.5153 - val_loss: 0.7009
Epoch 3/20
7/7 ━━━━━━━━━━━━━━━━━━━━ 0s 24ms/step - accuracy: 0.5550 - loss: 0.6559 - val_accuracy: 0.5203 - val_loss: 0.6649
Epoch 4/20
7/7 ━━━━━━━━━━━━━━━━━━━━ 0s 22ms/step - accuracy: 0.5750 - loss: 0.6251 - val_accuracy: 0.5628 - val_loss: 0.6369
Epoch 5/20
7/7 ━━━━━━━━━━━━━━━━━━━━ 0s 23ms/step - accuracy: 0.6350 - loss: 0.6008 - val_accuracy: 0.6439 - val_loss: 0.6135
Epoch 6/20
7/7 ━━━━━━━━━━━━━━━━━━━━ 0s 24ms/step - accuracy: 0.7300 - loss: 0.5799 - val_accuracy: 0.7201 - val_loss: 0.5938
Epoch 7/20
7/7 ━━━━━━━━━━━━━━━━━━━━ 0s 22ms/step - accuracy: 0.7900 - loss: 0.5619 - val_accuracy: 0.7824 - val_loss: 0.5765
Epoch 8/20
7/7 ━━━━━━━━━━━━━━━━━━━━ 0s 22ms/step - accuracy: 0.8400 - loss: 0.5458 - val_accuracy: 0.8180 - val_loss: 0.5609


In [201]:
model_B.evaluate(X_test_B, y_test_B)

63/63 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.9215 - loss: 0.4425


[0.4424670934677124, 0.921500027179718]

## ***Pretraining***

In [202]:
model_A = tf.keras.models.load_model("my_model_A.keras")

In [203]:
model_B_on_A = tf.keras.Sequential(model_A.layers[:-1])

In [204]:
model_B_on_A.add(tf.keras.layers.Dense(1,activation="sigmoid"))

In [205]:
model_A_clone = tf.keras.models.clone_model(model_A)
model_A_clone.set_weights(model_A.get_weights())

In [206]:
for layer in model_B_on_A.layers[:-1]:
  layer.trainable = False

In [207]:
optimizer = tf.keras.optimizers.SGD(learning_rate=0.001)
model_B_on_A.compile(loss="binary_crossentropy",optimizer=optimizer,metrics=["accuracy"])

In [208]:
history = model_B_on_A.fit(X_train_B,y_train_B,epochs=4,validation_data=(X_valid_B,y_valid_B))

Epoch 1/4
7/7 ━━━━━━━━━━━━━━━━━━━━ 1s 50ms/step - accuracy: 0.4750 - loss: 0.7570 - val_accuracy: 0.4936 - val_loss: 0.7410
Epoch 2/4
7/7 ━━━━━━━━━━━━━━━━━━━━ 0s 21ms/step - accuracy: 0.5250 - loss: 0.7336 - val_accuracy: 0.5242 - val_loss: 0.7221
Epoch 3/4
7/7 ━━━━━━━━━━━━━━━━━━━━ 0s 21ms/step - accuracy: 0.5450 - loss: 0.7120 - val_accuracy: 0.5598 - val_loss: 0.7043
Epoch 4/4
7/7 ━━━━━━━━━━━━━━━━━━━━ 0s 21ms/step - accuracy: 0.5950 - loss: 0.6919 - val_accuracy: 0.5994 - val_loss: 0.6876


In [209]:
for layer in model_B_on_A.layers[:-1]:
  layer.trainable = True
optimizer = tf.keras.optimizers.SGD(learning_rate=0.001)
model_B_on_A.compile(loss="binary_crossentropy",optimizer=optimizer,metrics=["accuracy"])
history = model_B_on_A.fit(X_train_B,y_train_B,epochs=20,validation_data=(X_valid_B,y_valid_B))
model_B_on_A.evaluate(X_test_B,y_test_B)

Epoch 1/20
7/7 ━━━━━━━━━━━━━━━━━━━━ 1s 51ms/step - accuracy: 0.6450 - loss: 0.6520 - val_accuracy: 0.7122 - val_loss: 0.6214
Epoch 2/20
7/7 ━━━━━━━━━━━━━━━━━━━━ 0s 21ms/step - accuracy: 0.7550 - loss: 0.5842 - val_accuracy: 0.7676 - val_loss: 0.5679
Epoch 3/20
7/7 ━━━━━━━━━━━━━━━━━━━━ 0s 21ms/step - accuracy: 0.8050 - loss: 0.5295 - val_accuracy: 0.8101 - val_loss: 0.5239
Epoch 4/20
7/7 ━━━━━━━━━━━━━━━━━━━━ 0s 23ms/step - accuracy: 0.8450 - loss: 0.4847 - val_accuracy: 0.8388 - val_loss: 0.4872
Epoch 5/20
7/7 ━━━━━━━━━━━━━━━━━━━━ 0s 22ms/step - accuracy: 0.8600 - loss: 0.4472 - val_accuracy: 0.8526 - val_loss: 0.4561
Epoch 6/20
7/7 ━━━━━━━━━━━━━━━━━━━━ 0s 22ms/step - accuracy: 0.8850 - loss: 0.4156 - val_accuracy: 0.8655 - val_loss: 0.4291
Epoch 7/20
7/7 ━━━━━━━━━━━━━━━━━━━━ 0s 22ms/step - accuracy: 0.8950 - loss: 0.3881 - val_accuracy: 0.8793 - val_loss: 0.4056
Epoch 8/20
7/7 ━━━━━━━━━━━━━━━━━━━━ 0s 22ms/step - accuracy: 0.9000 - loss: 0.3642 - val_accuracy: 0.8863 - val_loss: 0.3850


[0.2565724551677704, 0.9334999918937683]

In [210]:
model_B.evaluate(X_test_B,y_test_B)

63/63 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.9215 - loss: 0.4425


[0.4424670934677124, 0.921500027179718]

# **Momentum**

In [211]:
for layer in model_B_on_A.layers[:-1]:
  layer.trainable = True
optimizer = tf.keras.optimizers.SGD(learning_rate=0.001,momentum=0.5)
model_B_on_A.compile(loss="binary_crossentropy",optimizer=optimizer,metrics=["accuracy"])
history = model_B_on_A.fit(X_train_B,y_train_B,epochs=20,validation_data=(X_valid_B,y_valid_B))


Epoch 1/20
7/7 ━━━━━━━━━━━━━━━━━━━━ 1s 53ms/step - accuracy: 0.9400 - loss: 0.2167 - val_accuracy: 0.9238 - val_loss: 0.2492
Epoch 2/20
7/7 ━━━━━━━━━━━━━━━━━━━━ 0s 21ms/step - accuracy: 0.9400 - loss: 0.2067 - val_accuracy: 0.9248 - val_loss: 0.2399
Epoch 3/20
7/7 ━━━━━━━━━━━━━━━━━━━━ 0s 21ms/step - accuracy: 0.9400 - loss: 0.1976 - val_accuracy: 0.9298 - val_loss: 0.2318
Epoch 4/20
7/7 ━━━━━━━━━━━━━━━━━━━━ 0s 21ms/step - accuracy: 0.9450 - loss: 0.1898 - val_accuracy: 0.9337 - val_loss: 0.2248
Epoch 5/20
7/7 ━━━━━━━━━━━━━━━━━━━━ 0s 22ms/step - accuracy: 0.9450 - loss: 0.1829 - val_accuracy: 0.9357 - val_loss: 0.2186
Epoch 6/20
7/7 ━━━━━━━━━━━━━━━━━━━━ 0s 24ms/step - accuracy: 0.9450 - loss: 0.1769 - val_accuracy: 0.9387 - val_loss: 0.2132
Epoch 7/20
7/7 ━━━━━━━━━━━━━━━━━━━━ 0s 22ms/step - accuracy: 0.9450 - loss: 0.1716 - val_accuracy: 0.9407 - val_loss: 0.2083
Epoch 8/20
7/7 ━━━━━━━━━━━━━━━━━━━━ 0s 22ms/step - accuracy: 0.9450 - loss: 0.1669 - val_accuracy: 0.9416 - val_loss: 0.2040


In [212]:
model_B_on_A.evaluate(X_test_B,y_test_B)

63/63 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.9460 - loss: 0.1762


[0.17619675397872925, 0.9459999799728394]